In [4]:
import glob
import os

LANDING_ZONE = "/opt/airflow/data/raw_northwind_pings/"
# LANDING_ZONE = "/home/jovyan/work/data/raw_northwind_pings/"


files = glob.glob(LANDING_ZONE + "*.json") + glob.glob(LANDING_ZONE + "*.tmp")

for f in files:
    os.remove(f)

print(f"Deleted {len(files)} files from landing zone")

Deleted 8 files from landing zone


In [5]:
import shutil

if os.path.exists("/tmp/raw_northwind_pings/"):
    shutil.rmtree("/tmp/raw_northwind_pings/")

In [3]:
import pandas as pd
import time
import os
from datetime import datetime

# ─────────────────────────────────────────────
# LANDING ZONE
# ─────────────────────────────────────────────
# LANDING_ZONE = "/home/jovyan/work/data/raw_northwind_pings/"
LANDING_ZONE = "/opt/airflow/data/raw_northwind_pings/"
os.makedirs(LANDING_ZONE, exist_ok=True)

# ─────────────────────────────────────────────
# DATA PATHS
# ─────────────────────────────────────────────
DATA_DIR = "/opt/airflow/data/"

TABLES = {
    "orders":        DATA_DIR + "orders_table.csv",
    "order_details": DATA_DIR + "ordersdetails_table.csv",
    "customers":     DATA_DIR + "customers_table.csv",
    "products":      DATA_DIR + "products_table.csv",
    "employees":     DATA_DIR + "employees_table.csv",
    "categories":    DATA_DIR + "categories_table.csv",
    "suppliers":     DATA_DIR + "suppliers_table.csv",
    "shippers":      DATA_DIR + "shippers_table.csv"
}

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
BATCH_SIZE = 200
SLEEP_SECONDS = 120

# ─────────────────────────────────────────────
# LOAD TABLES
# ─────────────────────────────────────────────
dataframes = {}

print("\nLoading tables...")

for name, path in TABLES.items():
    try:
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()
        dataframes[name] = df
        print(f"{name:<15} → {len(df)} rows")
    except Exception as e:
        print(f"Failed {name}: {e}")

# ─────────────────────────────────────────────
# PREPARE POINTERS
# ─────────────────────────────────────────────
indices = {name: 0 for name in dataframes}

print("\nParallel streaming started...\n")

round_num = 1

while True:
    active = False

    print(f"\nRound {round_num}")

    for table_name, df in dataframes.items():

        start = indices[table_name]
        end = start + BATCH_SIZE

        if start >= len(df):
            continue

        active = True

        chunk = df.iloc[start:end].copy()
        chunk["ingestion_time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        file_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        file_name = f"{table_name}_batch_{file_id}.json"
        file_path = os.path.join(LANDING_ZONE, file_name)

        chunk.to_json(file_path, orient="records", lines=True)

        print(f"{table_name:<15} → {len(chunk)} rows")

        # تحديث المؤشر
        indices[table_name] = end

    if not active:
        break

    round_num += 1
    time.sleep(SLEEP_SECONDS)




Loading tables...
orders          → 894 rows
order_details   → 4309 rows
customers       → 95 rows
products        → 77 rows
employees       → 9 rows
categories      → 8 rows
suppliers       → 29 rows
shippers        → 3 rows

Parallel streaming started...


Round 1
orders          → 200 rows
order_details   → 200 rows
customers       → 95 rows
products        → 77 rows
employees       → 9 rows
categories      → 8 rows
suppliers       → 29 rows
shippers        → 3 rows


KeyboardInterrupt: 

In [34]:
# ─────────────────────────────────────────────
#  OPTIONAL: Inspect landing zone files
# ─────────────────────────────────────────────
import glob

files = sorted(glob.glob(LANDING_ZONE + "*.json"))
print(f"Files in landing zone: {len(files)}")

if files:
    print("\nSample record from latest batch:")
    sample = pd.read_json(files[-1], lines=True)
    print(sample.head(2).to_string())

Files in landing zone: 33

Sample record from latest batch:
   SupplierID                 CompanyName       ContactName         ContactTitle         Address         City Region PostalCode Country           Phone   Fax HomePage      ingestion_time
0           1              Exotic Liquids  Charlotte Cooper   Purchasing Manager  49 Gilbert St.       London   None    EC1 4SD      UK  (171) 555-2222  None     None 2026-05-06 13:27:03
1           2  New Orleans Cajun Delights     Shelley Burke  Order Administrator  P.O. Box 78934  New Orleans     LA      70117     USA  (100) 555-4822  None     None 2026-05-06 13:27:03
